In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import re
from datetime import datetime


In [2]:
rank_list = []
title_list = []
rating_list = []
release_date_list = []
genre_list = []
runtime_list = []
director_list = []
stars_list = []
country_list = []
language_list = []
budget_list = []

In [3]:
def add_up(movie_list, value):
    movie_list.append(value)

In [ ]:
driver = webdriver.Chrome()
driver.get('https://m.imdb.com/chart/top/')
i = 0

for i in range(1, 251):
    time.sleep(5)
    
    rank_title = driver.find_element(By.XPATH,f"/html/body/div[2]/main/div/div[3]/section/div/div[2]/div/ul/li[{i}]/div[2]/div/div/div[1]/a/h3").text
    rank_title_text = rank_title.split('.')
    rank = rank_title_text[0]
    
    title = rank_title_text[1].strip()
    
    rating_text = driver.find_element(By.XPATH, f'/html/body/div[2]/main/div/div[3]/section/div/div[2]/div/ul/li[{i}]/div[2]/div/div/span/div/span/span').text
    rating = rating_text[2:-1]
    
    stars_text = driver.find_element(By.XPATH, f'/html/body/div[2]/main/div/div[3]/section/div/div[2]/div/ul/li[{i}]/div[2]/div/div/span/div/span').text
    stars = stars_text.split(' ')[0]
    
    runtime_text = driver.find_element(By.XPATH, f'/html/body/div[2]/main/div/div[3]/section/div/div[2]/div/ul/li[{i}]/div[2]/div/div/div[2]/span[2]').text
    runtime = runtime_text

    link = driver.find_element(By.XPATH, f'/html/body/div[2]/main/div/div[3]/section/div/div[2]/div/ul/li[{i}]/div[2]/div/div/div[1]/a')
    driver.execute_script("arguments[0].scrollIntoView(true);", link)
    time.sleep(1)
    link.click()
    
    y = 1000
    for timer in range(0, 6):
        driver.execute_script("window.scrollTo(0, " + str(y) + ")")
        y += 1000
        time.sleep(2)

    release_date_text = driver.find_element(By.XPATH, "//li[@data-testid='title-details-releasedate']//div[@class='ipc-metadata-list-item__content-container']").text
    release_date = release_date_text.split(' (')[0]
    release_dates = datetime.strptime(release_date, "%B %d, %Y")
    formatted_date = release_dates.strftime("%Y-%m-%d")

    genre_texts = driver.find_element(By.XPATH, "//li[@data-testid='storyline-genres']//div[@class='ipc-metadata-list-item__content-container']").text
    genre_text = re.sub(r"([a-zA-Z])(?=[A-Z])", r"\1, ",genre_texts)

    director_text = driver.find_element(By.XPATH,"//body/div[@id='__next']/main[contains(@role,'main')]/div[contains(@role,'presentation')]/section[contains(@class,'ipc-page-background ipc-page-background--base sc-c41b9732-0 NeSef')]/div[contains(@role,'presentation')]/section[contains(@class,'ipc-page-background ipc-page-background--base sc-978e9339-0 ikfOqr')]/div[contains(@class,'ipc-page-grid ipc-page-grid--bias-left')]/div[contains(@class,'sc-978e9339-1 ihWZgK ipc-page-grid__item ipc-page-grid__item--span-2')]/section[contains(@class,'celwidget')]/ul[@role='presentation']/li[1]/div[1]").text
    director = director_text

    country_texts = driver.find_element(By.XPATH, "//li[@data-testid='title-details-origin']//div[@class='ipc-metadata-list-item__content-container']").text
    country_text = re.sub(r"([a-zA-Z])(?=[A-Z])", r"\1, ", country_texts)
    country = country_text

    language_text = driver.find_element(By.XPATH, "//li[@data-testid='title-details-languages']//div[@class='ipc-metadata-list-item__content-container']").text
    language = output_string = re.sub(r"([a-zA-Z])(?=[A-Z])", r"\1, ", language_text)

    try:
        budget_text = driver.find_element(By.XPATH, "//li[@data-testid='title-boxoffice-budget']//div[@class='ipc-metadata-list-item__content-container']").text
    except:
        budget_text = 'Null'
    budget = budget_text

    add_up(rank_list, rank)
    add_up(title_list, title)
    add_up(rating_list, rating)
    add_up(stars_list, stars)
    add_up(release_date_list, formatted_date)
    add_up(genre_list, genre_text)
    add_up(runtime_list, runtime)
    add_up(director_list, director)
    add_up(country_list, country)
    add_up(language_list, language)
    add_up(budget_list, budget)

    print('rank', rank, '\n title', title, '\n rating', rating, '\n release_Date', formatted_date, '\n genre',
          genre_text)
    print('runtime', runtime, '\n director', director, '\n stars', stars, '\n origin', country, '\n language', language)
    print('budget', budget)
    driver.back()


In [ ]:
star_list = []

In [18]:
for i in range(250):
    x = stars_list[i].replace('\n','')
    star_list.append(x)
    

In [19]:
import sqlite3

In [20]:
conn = sqlite3.connect('imdb.db')
cursor = conn.cursor() 

In [21]:
cursor.execute('''CREATE TABLE IF NOT EXISTS imdb_table( Rank TEXT, Title TEXT, Rating TEXT, Release_date TEXT, Genre TEXT, Runtime TEXT, Director TEXT, Stars TEXT, Country_of_origin TEXT, Language TEXT, Budget TEXT) ''')
for i in range(len(rank_list)):
    cursor.execute('INSERT INTO imdb_table(Rank, Title, Rating, Release_date, Genre, Runtime, Director, Stars, Country_of_origin, Language, Budget) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)',(rank_list[i], title_list[i], rating_list[i], release_date_list[i], genre_list[i], runtime_list[i], director_list[i], stars_list[i], country_list[i], language_list[i], budget_list[i]))
conn.commit()
conn.close()